**Author**: Felipe Matheus
**Purpose**: Experiment launcher ("control panel") for the annealing_iacs surrogate.

This notebook does NOT contain pipeline logic. All the logic (Model A -> OOF ->
NNLS -> Model B -> calibration -> metrics -> persistence) lives in
`src/modeling/Experiments.py`, which orchestrates the existing `Modeling` and
`Evaluation` helpers. Here you only:

1. Load and prepare the data (once).
2. Define a base `ExperimentConfig`.
3. Define the grid of variations you want to sweep.
4. Run and inspect the central `experiments_log.csv`.

Results layout on disk:
```
models/annealing_iacs/experiments/
    experiments_log.csv          <- 1 row per run (the "results spreadsheet")
    <tag>__<hash>/               <- 1 folder per run
        config.yaml
        model_a/   model_b/
        artifacts.pkl
        leaderboard_autogluon.csv
```

# 1. Setup

In [9]:
import logging
import os
import sys

import numpy as np
import pandas as pd

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

proc = Processing()
feng = FeatureEngineering()
mdl = Modeling()
evl = Evaluation()
runner = ExperimentRunner(mdl, evl, models_root="../../models")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 2. Data (same preparation as annealing_iacs.ipynb, run once)

In [10]:
SCHEMA_DATE = "080626"
PATH_DATA_RAW = "../../data/raw"
FILE_NAME = "dataset_annealing_iacs.csv"
FILE_NAME_SCHEMA_DATA = f"schema_annealing_essays_{SCHEMA_DATE}.csv"

TARGET = "iacs_final"
ALL_FEATURES = ["purity", "iacs", "temperature", "time"]

# ---- Schema (essay) data: explicit is_essay marker ----
df_raw_schema = pd.read_csv(os.path.join(PATH_DATA_RAW, FILE_NAME_SCHEMA_DATA))
df_schema = df_raw_schema[ALL_FEATURES + [TARGET]].dropna()
df_schema["is_essay"] = True

# ---- Literature data ----
df_raw = pd.read_csv(os.path.join(PATH_DATA_RAW, FILE_NAME))
df_float = proc.df_to_float(df_raw, drop_cols=["DOI"], ignore_columns=["material"])
df_labeled = feng.label_element(df_float).drop_duplicates()
df_with_masks = feng.add_ratio_mask_column(
    feng.add_ratio_mask_column(df_labeled, "grain_size"), "iacs",
)
df_lit = df_with_masks[df_with_masks.has_Cu == True][ALL_FEATURES + [TARGET]]
df_lit["is_essay"] = False

assert df_lit[ALL_FEATURES + [TARGET]].isna().sum().sum() == 0, "NaNs in inputs"

# ---- Concat. weight_col is built INSIDE the runner from is_essay + config ----
df = pd.concat([df_schema, df_lit], ignore_index=True)
print(f"Dataset: {df.shape} | essays: {df.is_essay.sum()} | lit: {(~df.is_essay).sum()}")
df.head()

Dataset: (96, 6) | essays: 9 | lit: 87


c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(
c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(


,purity,iacs,temperature,time,iacs_final,is_essay
0,99.9,88.260,623.0,30.0,88.350,True
1,99.9,88.260,623.0,60.0,88.655,True
2,99.9,88.260,623.0,90.0,88.800,True
3,99.9,99.825,573.0,30.0,102.340,True
4,99.9,99.390,573.0,30.0,102.230,True


# 3. Base config

In [11]:
base = ExperimentConfig(
    process="annealing_iacs",
    tag="annealing-v2",
    target=TARGET,
    features=tuple(ALL_FEATURES),
    # everything else uses the defaults; override here if needed, e.g.:
    # time_limit_a=120, weight_on_essay_rows=1.0, use_shared_folds=False,
)
print(base.run_id)

annealing-v2__a793edd1


# 4. Single run (sanity check before any grid)

Always run the base config alone first. Then run it 2-3 more times with
`tag="annealing-v1-rep2"` etc. to measure run-to-run noise: AutoGluon under a
time budget is NOT deterministic, and at n~90 this noise is the floor below
which grid differences mean nothing.

In [12]:
result = runner.run_experiment(df, base)
result["artifacts"]["metrics"]

2026-06-12 15:01:06,920 | INFO | src.modeling.Experiments | === Running annealing-v2__a793edd1 ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       4.53 GB / 31.57 GB (14.4%)
Disk Space Avail:   751.82 GB / 932.08 GB (80.7%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report weighted metrics.
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "c:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v2__a793edd1\mod

{'rmse': 4.092799343065013,
 'mae': 2.4994647377568566,
 'mape': 3.2498413833223023,
 'r2': 0.8800930662571467}

# 5. Grid

Keys are `ExperimentConfig` field names; values are lists of variants.
`features` variants must be tuples. Already-completed runs are skipped
(`force=True` to redo).

In [ ]:
grid = {
    "time_limit_a": [600],
    "num_bag_folds_a": [5, 10],
    "num_bag_sets_a": [1, 2],
    "weight_on_essay_rows": [1.0, 3.0],
    "features": [
        ("purity", "iacs", "temperature", "time")
    ],
}
log = runner.run_grid(df, base, grid)   # 3 x 2 x 2 = 12 runs
log

2026-06-12 15:04:37,529 | INFO | src.modeling.Experiments | Grid: 2 runs over ['time_limit_a', 'weight_on_essay_rows', 'features']
2026-06-12 15:04:37,534 | INFO | src.modeling.Experiments | === Running annealing-v2__time_limit_a=600__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__d46f25b3 ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       6.01 GB / 31.57 GB (19.0%)
Disk Space Avail:   751.77 GB / 932.08 GB (80.7%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,cfg_process,cfg_tag,cfg_target,cfg_features,cfg_weight_on_essay_rows,...,r2,cov_0.5,cov_0.8,cov_0.9,cov_0.95,c_opt,pct_truncated_aleat,nnls_recovery_ok,mean_sigma_epist,mean_sigma_aleat
0,annealing-v1__a793edd1,2026-06-12T13:18:37,135.1,96,318f1408,annealing_iacs,annealing-v1,iacs_final,purity|iacs|temperature|time,1.0,...,0.88569,0.6354,0.8333,0.9062,0.9479,1.4249,47.92,True,1.67132,1.19368
1,annealing-v1__time_limit_a=60__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__0a03f767,2026-06-12T13:20:28,110.6,96,318f1408,annealing_iacs,annealing-v1__time_limit_a=60__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time,iacs_final,purity|iacs|temperature|time,1.0,...,0.88009,0.6042,0.7812,0.9062,0.9479,1.5902,34.38,True,1.22183,1.22946
2,annealing-v1__time_limit_a=60__weight_on_essay_rows=1.0__features=iacs-temperature-time__e63c831d,2026-06-12T13:22:20,111.6,96,c826d116,annealing_iacs,annealing-v1__time_limit_a=60__weight_on_essay_rows=1.0__features=iacs-temperature-time,iacs_final,iacs|temperature|time,1.0,...,0.85386,0.6354,0.8333,0.9688,0.9688,2.0451,14.58,True,0.66274,1.57984
3,annealing-v1__time_limit_a=60__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time__17f60b62,2026-06-12T13:24:15,114.8,96,318f1408,annealing_iacs,annealing-v1__time_limit_a=60__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time,iacs_final,purity|iacs|temperature|time,3.0,...,0.87789,0.5938,0.8021,0.9062,0.9479,1.5782,34.38,True,1.31125,1.28751
4,annealing-v1__time_limit_a=60__weight_on_essay_rows=3.0__features=iacs-temperature-time__f4cbf931,2026-06-12T13:26:08,113.4,96,c826d116,annealing_iacs,annealing-v1__time_limit_a=60__weight_on_essay_rows=3.0__features=iacs-temperature-time,iacs_final,iacs|temperature|time,3.0,...,0.85417,0.5521,0.7500,0.8958,0.9271,1.6491,20.83,True,0.72908,1.56042
5,annealing-v1__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__a793edd1,2026-06-12T13:28:29,140.5,96,318f1408,annealing_iacs,annealing-v1__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time,iacs_final,purity|iacs|temperature|time,1.0,...,0.88569,0.6354,0.8333,0.9062,0.9479,1.4249,47.92,True,1.67132,1.19368
6,annealing-v1__time_limit_a=120__weight_on_essay_rows=1.0__features=iacs-temperature-time__0489fac0,2026-06-12T13:30:46,137.2,96,c826d116,annealing_iacs,annealing-v1__time_limit_a=120__weight_on_essay_rows=1.0__features=iacs-temperature-time,iacs_final,iacs|temperature|time,1.0,...,0.85680,0.6354,0.8125,0.8958,0.9167,2.0451,38.54,True,1.37377,1.26035
7,annealing-v1__time_limit_a=120__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time__acd86475,2026-06-12T13:33:13,147.0,96,318f1408,annealing_iacs,annealing-v1__time_limit_a=120__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time,iacs_final,purity|iacs|temperature|time,3.0,...,0.88848,0.5729,0.8542,0.9062,0.9375,1.3604,39.58,True,1.67718,1.33432
8,annealing-v1__time_limit_a=120__weight_on_essay_rows=3.0__features=iacs-temperature-time__ab29e4cd,2026-06-12T13:35:35,142.1,96,c826d116,annealing_iacs,annealing-v1__time_limit_a=120__weight_on_essay_rows=3.0__features=iacs-temperature-time,iacs_final,iacs|temperature|time,3.0,...,0.85966,0.5208,0.7708,0.9167,0.9688,1.6032,36.46,True,1.34159,1.58210
9,annealing-v1__time_limit_a=300__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__b3191dc7,2026-06-12T13:37:56,140.9,96,318f1408,annealing_iacs,annealing-v1__time_limit_a=300__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time,iacs_final,purity|iacs|temperature|time,1.0,...,0.88569,0.6354,0.8333,0.9062,0.9479,1.4249,47.92,True,1.67132,1.19368


# 6. Inspect results

In [14]:
log = runner.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_weight_on_essay_rows", "cfg_features",
    "rmse", "mae", "cov_0.9", "c_opt", "pct_truncated_aleat",
    "mean_sigma_epist", "mean_sigma_aleat", "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_weight_on_essay_rows,cfg_features,rmse,mae,cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
7,annealing-v1__time_limit_a=120__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time__acd86475,120,3.0,purity|iacs|temperature|time,3.94714,2.54604,0.9062,1.3604,39.58,1.67718,1.33432,147.0
11,annealing-v1__time_limit_a=300__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time__2a1bd50d,300,3.0,purity|iacs|temperature|time,3.94714,2.54604,0.9062,1.3604,39.58,1.67718,1.33432,146.9
15,annealing-v2__time_limit_a=600__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time__b61a3366,600,3.0,purity|iacs|temperature|time,3.94714,2.54604,0.9062,1.3604,39.58,1.67718,1.33432,311.2
0,annealing-v1__a793edd1,120,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,1.4249,47.92,1.67132,1.19368,135.1
9,annealing-v1__time_limit_a=300__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__b3191dc7,300,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,1.4249,47.92,1.67132,1.19368,140.9
14,annealing-v2__time_limit_a=600__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__d46f25b3,600,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,1.4249,47.92,1.67132,1.19368,308.9
5,annealing-v1__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__a793edd1,120,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,1.4249,47.92,1.67132,1.19368,140.5
1,annealing-v1__time_limit_a=60__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__0a03f767,60,1.0,purity|iacs|temperature|time,4.09280,2.49946,0.9062,1.5902,34.38,1.22183,1.22946,110.6
13,annealing-v2__a793edd1,120,1.0,purity|iacs|temperature|time,4.09280,2.49946,0.9062,1.5217,34.38,1.22183,1.33777,209.9
3,annealing-v1__time_limit_a=60__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time__17f60b62,60,3.0,purity|iacs|temperature|time,4.13020,2.54871,0.9062,1.5782,34.38,1.31125,1.28751,114.8


In [15]:
# Quick pivot: effect of one knob, marginalised over the others.
# Remember: compare against run-to-run noise (Section 4) before concluding.
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse                 mae             cov_0.9          
                      mean       std      mean       std      mean       std
cfg_time_limit_a                                                            
60                4.313747  0.234042  2.701742  0.206368  0.919250  0.033395
120               4.155462  0.233651  2.666198  0.241028  0.906217  0.006609
300               4.210962  0.277685  2.749553  0.262741  0.906225  0.008532
600               3.971630  0.034634  2.522780  0.032895  0.906200  0.000000

# 7. Load a winner for deployment / further analysis

Each run folder is self-contained: predictors + artifacts.pkl with weights,
`recalibration_c`, calibration tables.

In [16]:
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

RUN_ID = log.sort_values("rmse").iloc[0]["run_id"]
run_dir = Path("../../models/annealing_iacs/experiments") / RUN_ID

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-v1__time_limit_a=120__weight_on_essay_rows=3.0__features=purity-iacs-temperature-time__acd86475


,alpha,empirical_coverage,gap
0,0.50,0.572917,0.072917
1,0.80,0.854167,0.054167
2,0.90,0.906250,0.006250
3,0.95,0.937500,-0.012500
